# CASMI26 validation + learned reranker

Hides 1-2 train spectra of ~3000 molecules (the *seen-in-library* scenario that most test molecules fall into),
retrieves with the same code as the inference notebook, and compares rankings by top-1 / top-5 / top-25 / MRR:
current v3 fusion, spectral-only, plain vs modified cosine, ppm tie-break, and learned logistic / gradient-boosting rerankers (5-fold, split by molecule).

It prints the hit rate by best-cosine bucket (where does the current ranking lose?), then saves `reranker.joblib`.
Upload that file as a Kaggle dataset named `casmi26-reranker` and attach it to the inference notebook.

Limits: one or two query spectra per molecule here vs several in the real test set, so numbers are conservative; molecules with no sibling spectrum in train are not covered. Runtime is roughly 10-15 minutes.

In [ ]:
!pip install -q --no-index --find-links /kaggle/input/datasets/eveliaveldrine/rdkit-wheels rdkit==2025.3.6

In [ ]:
import os, gc, io, re, json, time, math, random, warnings, threading, queue
from collections import defaultdict
from functools import lru_cache

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

from rdkit import RDLogger, DataStructs
from rdkit.Chem import (MolFromSmiles, MolToSmiles, MolToInchiKey,
                        rdMolDescriptors, rdFingerprintGenerator)
RDLogger.DisableLog("rdApp.*")
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ======================= GLOBAL TIME BUDGET =======================
# Everything is scheduled against one wall-clock budget. CheMM-R1 generation
# (the only open-ended step) stops early enough that ChemMRL + fusion + writing
# the submission always fit before the hard deadline.
NOTEBOOK_START = time.time()
HARD_DEADLINE_HOURS = 3.75                      # 15 min safety margin below 4 h
HARD_DEADLINE_SECONDS = HARD_DEADLINE_HOURS * 3600
POST_RESERVE_MIN = 20                           # kept for ChemMRL + fusion + write
CHEMM_STOP_AT = HARD_DEADLINE_SECONDS - POST_RESERVE_MIN * 60

def elapsed_min():
    return (time.time() - NOTEBOOK_START) / 60

def remaining_s():
    return HARD_DEADLINE_SECONDS - (time.time() - NOTEBOOK_START)

def deadline_check(phase, reserve_seconds=900):
    print(f"[{phase}] elapsed={elapsed_min():.1f}min remaining={remaining_s()/60:.1f}min")
    return remaining_s() > reserve_seconds

# ======================= PATHS =======================
COMP_DIRS = [
    "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra",
    "/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra",
    "./data", ".",
]
COMP_DIR = next((p for p in COMP_DIRS if os.path.exists(p)), COMP_DIRS[0])
TEST_PATH  = os.path.join(COMP_DIR, "test.parquet")
TRAIN_PATH = os.path.join(COMP_DIR, "train.parquet")
SAMPLE_SUB = os.path.join(COMP_DIR, "sample_submission.csv")
WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "."
SUBMISSION_PATH = os.path.join(WORKING_DIR, "submission.csv")
DEBUG_PATH = os.path.join(WORKING_DIR, "chemm_hybrid_debug.json")
CHEMM_CACHE_PATH = os.path.join(WORKING_DIR, "chemm_cache.jsonl")   # resume file

MODEL_ID = "/kaggle/input/datasets/eveliaveldrine/zzha6204-chemm-r1-offline"

# ======================= CheMM-R1 =======================
USE_CHEMM = True
CHEMM_MAX_GPUS = 2
CHEMM_BATCH_SIZE = 6              # molecules per generate() call, per GPU
CHEMM_MIN_TOKENS = 640            # token cap is chosen adaptively in [MIN, MAX]
CHEMM_MAX_TOKENS = 2048           #   from the time that is left (see generation cell)
CHEMM_RESCUE_TOKENS = 96          # if the cap cuts the reasoning, force "<answer>" and finish
CHEMM_TEMPERATURE = 0.6           # <= 0 means greedy
CHEMM_TOP_P = 0.95
CHEMM_TOP_K = 20
CHEMM_SKIP_IF_SPECTRAL_ABOVE = 0.95   # library match this good needs no LLM call
CHEMM_MOLECULES_CAP = None            # None = as many as the time budget allows

# ======================= ChemMRL =======================
USE_CHEMMRL = True
CHEMMRL_MODEL_ID = "Derify/ChemMRL"
CHEMMRL_BATCH_SIZE = 64
CHEMMRL_MAX_ANCHORS = 2
CHEMMRL_EMB_DIM = 256
CHEMMRL_LOCAL_PATHS = [
    "/kaggle/input/datasets/eveliaveldrine/derify-chemmrl-offline",
    "/kaggle/input/chem-mrl",
    "./chemmrl",
]

# ======================= Retrieval =======================
MAX_STRUCTURES_FOR_LIBRARY = 600_000
MAX_SPECTRA_PER_STRUCT = 3        # keep several train spectra per structure (different adducts/energies)
MASS_CANDIDATE_LIMIT = 400        # many isomers share one exact mass; a small cap drops true hits at random
MAX_QUERY_SPECTRA_SCORING = 8
PRIMARY_PPM = 10.0
FALLBACK_PPM = 30.0
FINAL_K = 25
MAX_QUERY_PEAKS = 128
PEAK_TOL = 0.02
STRONG_SPECTRAL = 0.70            # spectral match this strong is guaranteed a top slot

# ======================= Learned reranker (trained in the validation notebook) =======================
RERANKER_PATHS = [
    "/kaggle/input/datasets/eveliaveldrine/casmi26-reranker/reranker.joblib",
    "/kaggle/input/casmi26-reranker/reranker.joblib",
    "./reranker.joblib",
]
LEARNED_BOTH_BONUS = 0.30     # CheMM-R1 agrees with a retrieved structure (untuned)
LEARNED_NOVEL_BASE = 0.10     # CheMM-R1 structure not in the library: base score ...
LEARNED_NOVEL_SLACK = 0.50    # ... plus this x (1 - top retrieval probability)
LEARNED_ANCHOR_W   = 0.10     # similarity to CheMM anchors

# ======================= Fusion weights (used when no reranker file is found) =======================
W_MASS          = 0.15
W_SPECTRAL      = 0.35
W_CHEMMRL       = 0.20
W_PRIOR         = 0.05
BONUS_CHEMM_R1  = 0.20
SOURCE_BONUS = {"chemm_r1": 0.35, "train_retrieval": 0.0,
                "mass_fallback": -0.05, "nearest_mass": -0.05, "both": 0.40}

print("COMP_DIR:", COMP_DIR)
print("MODEL_ID:", MODEL_ID)
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {p.name} | VRAM {p.total_memory/1e9:.1f} GB | sm_{p.major}{p.minor}")

In [ ]:
ADDUCT_DELTAS = {
    "[M+H]+":          1.007276466621,
    "[M+NH4]+":        18.033823005,
    "[M-H2O+H]+":      -17.003288036,
    "[M-2H2O+H]+":     -35.013853101,
    "[M+Na]+":         22.989218,
    "[M+K]+":          38.963158,
    "[M-H]-":          -1.007276466621,
    "[M-H2O-H]-":      -19.0178415,
    "[M+CH2O2-H]-":    44.998201,
    "[M+Cl]-":         34.969402,
    "[M+FA-H]-":       44.998201,
    "[M+HCOO]-":       44.998201,
    "[M+Ac-H]-":       59.013851,
    "[M+CH3COO]-":     59.013851,
}

def neutral_mass_from_adduct(precursor_mz, adduct):
    adduct = str(adduct).strip()
    if adduct not in ADDUCT_DELTAS:
        raise ValueError(f"Unsupported CASMI adduct: {adduct}")
    return float(precursor_mz) - ADDUCT_DELTAS[adduct]

def safe_neutral_mass(precursor_mz, adduct):
    try:
        return neutral_mass_from_adduct(precursor_mz, adduct)
    except Exception:
        return np.nan

def ppm_error(mass, target):
    return abs(float(mass) - float(target)) / max(abs(float(target)), 1e-9) * 1e6

# Plain InChIKey connectivity block, cached. (The old tautomer-canonicalising key was
# the slowest step in the notebook -- it ran for every candidate of every molecule --
# and it did not match the InChIKey14 the scorer computes from the submitted SMILES.)
@lru_cache(maxsize=1_000_000)
def ikey14(smi):
    mol = MolFromSmiles(smi) if isinstance(smi, str) else None
    if mol is None:
        return None
    try:
        k = MolToInchiKey(mol)
        return k.split("-")[0] if k else None
    except Exception:
        return None

def structure_prior_from_mol(mol):
    heavy = mol.GetNumHeavyAtoms()
    rings = rdMolDescriptors.CalcNumRings(mol)
    hetero = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() in (7, 8, 15, 16, 17))
    return float(0.002 * min(heavy, 80) + 0.01 * min(rings, 8) + 0.003 * min(hetero, 12))

print("Supported adducts:", len(ADDUCT_DELTAS))

In [ ]:
_EMPTY = (np.empty(0, np.float32), np.empty(0, np.float32))

def select_peaks(mzs, intensities, max_peaks=MAX_QUERY_PEAKS, noise_floor=0.005):
    """Clean a spectrum: finite/positive peaks, base-peak normalised, noise filtered,
    top-N by intensity, and ALWAYS sorted by m/z (the matcher needs sorted m/z; the old
    code only preserved input order)."""
    if mzs is None or intensities is None:
        return _EMPTY
    mzs = np.asarray(mzs, dtype=np.float64)
    ints = np.asarray(intensities, dtype=np.float64)
    n = min(mzs.size, ints.size)
    if n == 0:
        return _EMPTY
    mzs, ints = mzs[:n], ints[:n]
    ok = np.isfinite(mzs) & np.isfinite(ints) & (ints > 0)
    mzs, ints = mzs[ok], ints[ok]
    if ints.size == 0:
        return _EMPTY
    ints = ints / ints.max()
    keep = ints >= noise_floor
    mzs, ints = mzs[keep], ints[keep]
    if mzs.size > max_peaks:
        top = np.argsort(-ints, kind="stable")[:max_peaks]
        mzs, ints = mzs[top], ints[top]
    order = np.argsort(mzs, kind="stable")
    return mzs[order].astype(np.float32), ints[order].astype(np.float32)

def to_weights(ints):
    """sqrt-intensity, L2-normalised (down-weights dominant peaks, standard for MS/MS cosine)."""
    w = np.sqrt(ints.astype(np.float64))
    n = np.linalg.norm(w)
    return (w / n).astype(np.float32) if n > 0 else w.astype(np.float32)

def _nearest(l_mz, x, tol):
    """For every x find the nearest peak in sorted l_mz; return (index, within_tol)."""
    n = l_mz.shape[0]
    pos = np.searchsorted(l_mz, x)
    lo = np.clip(pos - 1, 0, n - 1)
    hi = np.clip(pos, 0, n - 1)
    dlo = np.abs(l_mz[lo] - x)
    dhi = np.abs(l_mz[hi] - x)
    use_hi = dhi < dlo
    return np.where(use_hi, hi, lo), np.minimum(dlo, dhi) <= tol

def _score_parts(qi_parts, lj_parts, pr_parts):
    if not qi_parts:
        return 0.0
    qi = np.concatenate(qi_parts); lj = np.concatenate(lj_parts); pr = np.concatenate(pr_parts)
    if len(qi_parts) > 1 or np.unique(lj).size < lj.size:
        # greedy one-to-one assignment: best products first, each peak used once
        o = np.argsort(-pr, kind="stable"); qi, lj, pr = qi[o], lj[o], pr[o]
        _, f = np.unique(qi, return_index=True); lj, pr = lj[f], pr[f]
        o = np.argsort(-pr, kind="stable"); lj, pr = lj[o], pr[o]
        _, f = np.unique(lj, return_index=True); pr = pr[f]
    return float(min(pr.sum(), 1.0))

def spectral_similarity(q_mz, q_w, l_mz, l_w, shift=0.0, tol=PEAK_TOL, with_plain=False):
    """(Modified) cosine between two sorted spectra, vectorised, one-to-one matching.
    shift = precursor difference; peaks may match directly or shifted (modified cosine).
    with_plain=True returns (modified, plain) from a single matching pass."""
    if q_mz.size == 0 or l_mz.size == 0:
        return (0.0, 0.0) if with_plain else 0.0
    qi_parts, lj_parts, pr_parts = [], [], []
    j, ok = _nearest(l_mz, q_mz, tol)
    if ok.any():
        qi = np.flatnonzero(ok)
        qi_parts.append(qi); lj_parts.append(j[ok]); pr_parts.append(q_w[qi] * l_w[j[ok]])
    n_direct = len(qi_parts)
    if abs(shift) > tol:
        j2, ok2 = _nearest(l_mz, q_mz - shift, tol)
        if ok2.any():
            qi = np.flatnonzero(ok2)
            qi_parts.append(qi); lj_parts.append(j2[ok2]); pr_parts.append(q_w[qi] * l_w[j2[ok2]])
    mod = _score_parts(qi_parts, lj_parts, pr_parts)
    if not with_plain:
        return mod
    plain = mod if len(qi_parts) == n_direct else _score_parts(qi_parts[:n_direct], lj_parts[:n_direct], pr_parts[:n_direct])
    return mod, plain

# ---------------- spectrum image for CheMM-R1 (identical look to the original) ----------------
def render_ms2_png(args):
    mzs, intensities, precursor_mz, adduct = args
    fig, ax = plt.subplots(figsize=(7, 3.5), dpi=90)
    if len(mzs):
        ax.vlines(mzs, 0, intensities, linewidth=0.8)
    ax.axvline(precursor_mz, linestyle="--", linewidth=1.0)
    ax.set_xlabel("m/z"); ax.set_ylabel("relative intensity")
    ax.set_ylim(0, 1.08)
    ax.set_xlim(0, max(precursor_mz * 1.05, 100))
    ax.set_title(f"MS2 | {adduct} | precursor={precursor_mz:.4f}")
    ax.tick_params(labelsize=7)
    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    plt.close(fig)
    return buf.getvalue()

def png_to_image(b):
    return Image.open(io.BytesIO(b)).convert("RGB")

In [ ]:
# Stream the training parquet in batches (no giant DataFrame), keep up to
# MAX_SPECTRA_PER_STRUCT spectra per structure, store spectra as compact float32 arrays.
print("Building spectral library from train...")
t0 = time.time()
pf = pq.ParquetFile(TRAIN_PATH)
train_cols = set(pf.schema_arrow.names)
cols = ["inchikey14", "normalized_smiles", "ms2_mzs", "ms2_normalized_intensities"]
HAS_PREC = "precursor_mz" in train_cols
if HAS_PREC:
    cols.append("precursor_mz")      # enables modified-cosine matching across adducts
print(f"Train rows: {pf.metadata.num_rows:,} | precursor_mz available: {HAS_PREC}")

struct_index = {}                    # inchikey14 -> structure idx (-1 = unparsable)
lib_smiles, lib_ikey, lib_mass_l, lib_prior_l, struct_specs = [], [], [], [], []
spec_mz, spec_w, spec_prec = [], [], []

pbar = tqdm(total=pf.metadata.num_rows, desc="Indexing train")
for batch in pf.iter_batches(batch_size=20_000, columns=cols):
    df = batch.to_pandas()
    pbar.update(len(df))
    prec_col = df["precursor_mz"].to_numpy() if HAS_PREC else None
    for r, (ikey, smi, mzs, ints) in enumerate(zip(df["inchikey14"], df["normalized_smiles"],
                                                   df["ms2_mzs"], df["ms2_normalized_intensities"])):
        if not isinstance(smi, str) or not isinstance(ikey, str) or not ikey:
            continue
        idx = struct_index.get(ikey)
        if idx is None:
            if len(lib_smiles) >= MAX_STRUCTURES_FOR_LIBRARY:
                continue
            mol = MolFromSmiles(smi)
            if mol is None:
                struct_index[ikey] = -1
                continue
            try:
                mw = rdMolDescriptors.CalcExactMolWt(mol)
                prior = structure_prior_from_mol(mol)
            except Exception:
                struct_index[ikey] = -1
                continue
            idx = len(lib_smiles)
            struct_index[ikey] = idx
            lib_smiles.append(smi); lib_ikey.append(ikey)
            lib_mass_l.append(mw); lib_prior_l.append(prior); struct_specs.append([])
        elif idx < 0:
            continue
        if len(struct_specs[idx]) >= MAX_SPECTRA_PER_STRUCT:
            continue
        mz, it = select_peaks(mzs, ints)
        if len(mz) < 3:
            continue
        sid = len(spec_mz)
        spec_mz.append(mz); spec_w.append(to_weights(it))
        spec_prec.append(float(prec_col[r]) if HAS_PREC else float("nan"))
        struct_specs[idx].append(sid)
    del df
pbar.close()
del struct_index; gc.collect()

if not lib_smiles:
    raise RuntimeError("Library is empty - check TRAIN_PATH / column names")

# sort structures by mass for binary-search lookups
order = np.argsort(np.asarray(lib_mass_l, dtype=np.float64), kind="stable")
lib_mass  = np.asarray(lib_mass_l, dtype=np.float64)[order]
lib_prior = np.asarray(lib_prior_l, dtype=np.float32)[order]
lib_smiles = [lib_smiles[i] for i in order]
lib_ikey   = [lib_ikey[i] for i in order]
struct_specs = [struct_specs[i] for i in order]
spec_prec = np.asarray(spec_prec, dtype=np.float64)
del lib_mass_l, lib_prior_l, order; gc.collect()

n_with_spec = sum(1 for s in struct_specs if s)
print(f"Library: {len(lib_smiles):,} structures | {len(spec_mz):,} spectra "
      f"({n_with_spec:,} structures with >=1 spectrum) | {(time.time()-t0)/60:.1f} min")
print(f"Elapsed: {elapsed_min():.1f} min")

In [ ]:
_fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

@lru_cache(maxsize=200_000)
def get_fp(smi):
    m = MolFromSmiles(smi)
    return _fp_gen.GetFingerprint(m) if m is not None else None

def fp_similarity(smi_a, smi_b):
    a, b = get_fp(smi_a), get_fp(smi_b)
    if a is None or b is None:
        return 0.0
    return float(DataStructs.TanimotoSimilarity(a, b))

EXCLUDE_SIDS = set()      # spectrum ids hidden from matching (used only by the validation notebook)

def candidate_spectral_score(idx, qspecs):
    """Best modified cosine and best plain cosine between any train spectrum of this structure
    and any query spectrum; also returns how many train spectra were usable."""
    best_mod = best_plain = 0.0
    n_used = 0
    for sid in struct_specs[idx]:
        if sid in EXCLUDE_SIDS:
            continue
        n_used += 1
        lmz, lw, lp = spec_mz[sid], spec_w[sid], spec_prec[sid]
        for q in qspecs:
            shift = (q["prec"] - lp) if (np.isfinite(lp) and np.isfinite(q["prec"])) else 0.0
            m, p = spectral_similarity(q["mz"], q["w"], lmz, lw, shift, with_plain=True)
            if m > best_mod: best_mod = m
            if p > best_plain: best_plain = p
    return best_mod, best_plain, n_used

def _window(target_mass, ppm):
    d = target_mass * ppm * 1e-6
    lo = int(np.searchsorted(lib_mass, target_mass - d, side="left"))
    hi = int(np.searchsorted(lib_mass, target_mass + d, side="right"))
    return lo, hi

def _closest(lo, hi, target_mass, limit):
    idx = np.arange(lo, hi)
    if idx.size == 0:
        return idx
    return idx[np.argsort(np.abs(lib_mass[idx] - target_mass), kind="stable")[:limit]]

def retrieve_candidates(target_mass, qspecs):
    """Mass-window candidates from the library, scored by spectral similarity.
    Always tries to give >= FINAL_K candidates (widens to FALLBACK_PPM, then nearest mass)
    so the 25 guess slots are never wasted."""
    qs = sorted(qspecs, key=lambda s: -len(s["mz"]))[:MAX_QUERY_SPECTRA_SCORING]
    chosen = {}
    lo, hi = _window(target_mass, PRIMARY_PPM)
    for i in _closest(lo, hi, target_mass, MASS_CANDIDATE_LIMIT):
        chosen[int(i)] = "train_retrieval"
    if len(chosen) < FINAL_K:
        lo, hi = _window(target_mass, FALLBACK_PPM)
        for i in _closest(lo, hi, target_mass, MASS_CANDIDATE_LIMIT):
            if len(chosen) >= MASS_CANDIDATE_LIMIT:
                break
            chosen.setdefault(int(i), "mass_fallback")
    if not chosen:
        pos = int(np.searchsorted(lib_mass, target_mass))
        lo, hi = max(0, pos - FINAL_K), min(len(lib_mass), pos + FINAL_K)
        for i in _closest(lo, hi, target_mass, FINAL_K):
            chosen[int(i)] = "nearest_mass"
    out = []
    for i, src in chosen.items():
        mw = float(lib_mass[i])
        sm, sp, nu = candidate_spectral_score(i, qs) if qs else (0.0, 0.0, 0)
        out.append({
            "idx": i, "smiles": lib_smiles[i], "mass": mw,
            "ppm": ppm_error(mw, target_mass),
            "spectral_score": sm, "spectral_plain": sp, "n_lib_spec": nu,
            "prior_score": float(lib_prior[i]), "source": src,
        })
    return out

In [ ]:
# ChemMRL state (filled later; fusion works without it)
chem_mrl = None
CHEMMRL_ACTIVE = False
CHEMMRL_API = None
chem_mrl_embeddings = {}

def dense_tanimoto(a, b):
    dot = float(np.dot(a, b))
    denom = float(np.dot(a, a) + np.dot(b, b) - dot)
    return float(np.clip(dot / denom, 0.0, 1.0)) if denom > 1e-12 else 0.0

def anchor_similarity(smi, anchors):
    """Similarity of a candidate to CheMM-R1's own predictions: ChemMRL embedding Tanimoto
    when available, Morgan Tanimoto when the ChemMRL channel is off."""
    if not anchors:
        return 0.0
    if CHEMMRL_ACTIVE and chem_mrl_embeddings:
        v = chem_mrl_embeddings.get(smi)
        if v is None:
            return 0.0
        scores = [dense_tanimoto(v, chem_mrl_embeddings[a]) for a in anchors if a in chem_mrl_embeddings]
        return max(scores) if scores else 0.0
    return max(fp_similarity(smi, a) for a in anchors)

FEATURE_NAMES = ["spec_mod", "spec_plain", "spec_gap", "spec_rank", "ppm", "prior", "n_lib_spec", "log_ncand"]

def candidate_features(entries):
    """Per-candidate features for the learned reranker (same code is used for training and inference)."""
    n = len(entries)
    if n == 0:
        return np.zeros((0, len(FEATURE_NAMES)), dtype=np.float32)
    sm = np.array([e["spectral_score"] for e in entries], dtype=np.float64)
    sp = np.array([e.get("spectral_plain", 0.0) for e in entries], dtype=np.float64)
    order = np.argsort(-sm, kind="stable")
    rank = np.empty(n); rank[order] = np.arange(n)
    srt = sm[order]
    second = srt[1] if n > 1 else 0.0
    gap = sm - np.where(rank == 0, second, srt[0])          # margin over the best OTHER candidate
    F = np.column_stack([
        sm, sp, gap, np.minimum(rank, 50),
        np.array([e["ppm"] for e in entries]),
        np.array([e["prior_score"] for e in entries]),
        np.array([e.get("n_lib_spec", 0) for e in entries], dtype=np.float64),
        np.full(n, math.log(n)),
    ])
    return F.astype(np.float32)

RERANKER = None
for _p in RERANKER_PATHS:
    if os.path.exists(_p):
        try:
            import joblib
            _obj = joblib.load(_p)
            if list(_obj["features"]) == FEATURE_NAMES:
                RERANKER = _obj["model"]
                print(f"Learned reranker loaded: {_p}")
            else:
                print(f"[WARN] reranker feature mismatch ({_obj['features']}); using hand-tuned weights")
        except Exception as e:
            print(f"[WARN] could not load reranker {_p}: {e!r}; using hand-tuned weights")
        break
if RERANKER is None:
    print("No learned reranker found - using hand-tuned fusion weights")

def make_chemm_entry(smi, target_mass):
    mol = MolFromSmiles(smi)
    if mol is None:
        return None
    mw = rdMolDescriptors.CalcExactMolWt(mol)
    return {"idx": None, "smiles": MolToSmiles(mol), "mass": mw, "ppm": ppm_error(mw, target_mass),
            "spectral_score": 0.0, "prior_score": structure_prior_from_mol(mol), "source": "chemm_r1"}

def fuse_candidates(target_mass, pool, chemm_smiles, anchors):
    entries = [dict(e, chemm_exact=False) for e in pool]

    chemm_keys = {}
    for s in chemm_smiles:
        k = ikey14(s)
        if k:
            chemm_keys.setdefault(k, s)
    if chemm_keys:
        have = set()
        for e in entries:
            k = ikey14(e["smiles"]) or e["smiles"]
            e["key"] = k
            have.add(k)
            if k in chemm_keys:                      # CheMM-R1 agrees with a library structure
                e["source"] = "both"; e["chemm_exact"] = True
        for k, s in chemm_keys.items():
            if k in have:
                continue
            ce = make_chemm_entry(s, target_mass)
            if ce is None or ce["ppm"] > FALLBACK_PPM:   # wrong formula -> discard
                continue
            ce["key"] = k; ce["chemm_exact"] = True
            entries.append(ce)

    pool_entries = [e for e in entries if e["source"] != "chemm_r1"]
    learned = RERANKER is not None and len(pool_entries) > 0
    if learned:
        try:
            probs = RERANKER.predict_proba(candidate_features(pool_entries))[:, 1]
            for e, p in zip(pool_entries, probs):
                e["retr_prob"] = float(p)
            top_prob = float(probs.max())
        except Exception as ex:
            print(f"[WARN] reranker failed ({ex!r}); falling back to hand-tuned weights")
            learned = False

    for e in entries:
        e["mass_score"] = math.exp(-e["ppm"] / 5.0)
        e["anchor_score"] = anchor_similarity(e["smiles"], anchors)
        if learned:
            if e["source"] == "chemm_r1":      # structure only CheMM-R1 proposed
                base = LEARNED_NOVEL_BASE + LEARNED_NOVEL_SLACK * (1.0 - top_prob)
            else:
                base = e["retr_prob"] + (LEARNED_BOTH_BONUS if e["chemm_exact"] else 0.0)
            e["fusion_score"] = base + LEARNED_ANCHOR_W * e["anchor_score"]
            continue
        score = (W_MASS * e["mass_score"] + W_SPECTRAL * e["spectral_score"]
                 + W_CHEMMRL * e["anchor_score"] + W_PRIOR * e["prior_score"]
                 + SOURCE_BONUS.get(e["source"], 0.0))
        if e["chemm_exact"]:
            score += BONUS_CHEMM_R1
        if e["source"] in ("chemm_r1", "both") and e["ppm"] <= 5:
            score += BONUS_CHEMM_R1
        e["fusion_score"] = score

    ranked = sorted(entries, key=lambda x: x["fusion_score"], reverse=True)
    if learned:
        ordered, forced_ids = ranked, set()          # the learned scores already encode the trade-offs
    else:
        # guaranteed slots: top-3 CheMM-supported candidates + any very strong spectral match
        forced = [x for x in ranked if x["source"] in ("chemm_r1", "both") and x["ppm"] <= PRIMARY_PPM][:3]
        forced += [x for x in sorted(entries, key=lambda x: x["spectral_score"], reverse=True)[:2]
                   if x["spectral_score"] >= STRONG_SPECTRAL and x["ppm"] <= FALLBACK_PPM]
        forced_ids, forced_u = set(), []
        for x in forced:
            if id(x) not in forced_ids:
                forced_ids.add(id(x)); forced_u.append(x)
        forced_u.sort(key=lambda x: x["fusion_score"], reverse=True)
        ordered = forced_u + [x for x in ranked if id(x) not in forced_ids]

    selected, seen = [], set()
    for x in ordered:
        k = x.get("key") or ikey14(x["smiles"]) or x["smiles"]
        if k in seen:
            continue
        seen.add(k); x["key"] = k
        selected.append(x)
        if len(selected) >= FINAL_K:
            break
    return selected

def write_submission(final_records, path=SUBMISSION_PATH):
    sample_sub = pd.read_csv(SAMPLE_SUB)
    id_col = "molecule_id"
    target_col = "smiles" if "smiles" in sample_sub.columns else \
                 next(c for c in sample_sub.columns if c != id_col)
    rows = []
    for mid in sample_sub[id_col]:
        guesses, seen = [], set()
        for c in final_records.get(mid, []):
            k = c.get("key") or ikey14(c["smiles"]) or c["smiles"]
            if k in seen:
                continue
            seen.add(k); guesses.append(c["smiles"])
            if len(guesses) >= FINAL_K:
                break
        rows.append(";".join(guesses) if guesses else "CCO")
    sample_sub[target_col] = rows
    sample_sub.to_csv(path, index=False)
    return sample_sub, target_col

def run_fusion(chemm_preds, anchors_by_mid, desc="Fusion"):
    out = {}
    for mid, tm in tqdm(target_mass_by_mid.items(), desc=desc):
        out[mid] = fuse_candidates(float(tm), mass_pool_by_mid.get(mid, []),
                                   chemm_preds.get(mid, {}).get("smiles", []),
                                   anchors_by_mid.get(mid, []))
    return out

print("Fusion + submission helpers ready")

In [ ]:
# Validation design ("seen" scenario, which is what ~60% of the test set looks like):
# take structures that have >= 2 spectra in the library, HIDE 1-2 of their spectra, use the hidden
# spectra as the query, and check whether the ranking still puts the true structure first.
# Mass noise mimics the real error of the neutral mass estimate.
N_VAL = 3000
NOISE_PPM = 1.5
rng = np.random.default_rng(SEED)

eligible = [i for i, s in enumerate(struct_specs) if len(s) >= 2]
print(f"Structures with >= 2 spectra: {len(eligible):,} of {len(struct_specs):,}")
chosen = rng.choice(eligible, size=min(N_VAL, len(eligible)), replace=False)

val = []
for i in chosen:
    sids = struct_specs[i]
    k = 2 if len(sids) >= 3 else 1
    held = [int(x) for x in rng.choice(sids, size=k, replace=False)]
    val.append({"idx": int(i), "held": held})
EXCLUDE_SIDS.clear()
EXCLUDE_SIDS.update(sid for v in val for sid in v["held"])
RERANKER = None                       # baseline = current hand-tuned fusion
print(f"{len(val)} validation molecules | {len(EXCLUDE_SIDS)} hidden spectra | mass noise sigma {NOISE_PPM} ppm")

In [ ]:
import pandas as pd
t0 = time.time()
rows, base_rank = [], {}
for qi, v in enumerate(tqdm(val, desc="Validation retrieval")):
    qs = [{"mz": spec_mz[s], "w": spec_w[s], "prec": float(spec_prec[s])} for s in v["held"]]
    tm = float(lib_mass[v["idx"]]) * (1.0 + rng.normal(0.0, NOISE_PPM) * 1e-6)
    pool = retrieve_candidates(tm, qs)
    if not pool:
        base_rank[qi] = 10**6
        continue
    F = candidate_features(pool)
    # rank of the truth under the CURRENT hand-tuned fusion (top-25 list)
    fused = fuse_candidates(tm, pool, [], [])
    pos = [j for j, e in enumerate(fused) if e["idx"] == v["idx"]]
    base_rank[qi] = pos[0] if pos else 10**6
    for e, f in zip(pool, F):
        rows.append((qi, int(e["idx"] == v["idx"]), *f.tolist()))
df = pd.DataFrame(rows, columns=["q", "y"] + FEATURE_NAMES)
print(f"{len(df):,} candidate rows from {df.q.nunique()} queries | {(time.time()-t0)/60:.1f} min")
print(f"Truth inside the candidate pool: {df.groupby('q').y.max().mean()*100:.1f}% of queries")

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def metrics_from_scores(df, score, name):
    d = df.assign(s=score)
    d["r"] = d.groupby("q")["s"].rank(ascending=False, method="first") - 1
    truth = d[d.y == 1].set_index("q")["r"]
    allq = df.q.unique()
    r = truth.reindex(allq).fillna(10**6).to_numpy()
    return {"ranking": name, "top1": (r < 1).mean(), "top5": (r < 5).mean(),
            "top25": (r < 25).mean(), "MRR": float(np.mean(1.0 / (r + 1)))}

results = []
qs_all = df.q.unique()
br = np.array([base_rank[q] for q in qs_all])
results.append({"ranking": "current fusion (v3 weights)", "top1": (br < 1).mean(), "top5": (br < 5).mean(),
                "top25": (br < 25).mean(), "MRR": float(np.mean(1.0 / (br + 1)))})
results.append(metrics_from_scores(df, df.spec_mod.values, "spectral only (modified cosine)"))
results.append(metrics_from_scores(df, df.spec_plain.values, "spectral only (plain cosine)"))
results.append(metrics_from_scores(df, df.spec_mod.values - 0.01 * df.ppm.values, "spectral, ppm as tie-break"))

# out-of-fold learned rerankers (group split by query molecule)
X = df[FEATURE_NAMES].values; y = df.y.values; groups = df.q.values
gkf = GroupKFold(n_splits=5)
oof = {"logistic": np.zeros(len(df)), "hist-gbm": np.zeros(len(df))}
for tr, te in gkf.split(X, y, groups):
    lr = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000, class_weight=None))
    lr.fit(X[tr], y[tr]); oof["logistic"][te] = lr.predict_proba(X[te])[:, 1]
    gb = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.08, max_iter=250, random_state=SEED)
    gb.fit(X[tr], y[tr]); oof["hist-gbm"][te] = gb.predict_proba(X[te])[:, 1]
for k, sc in oof.items():
    results.append(metrics_from_scores(df, sc, f"learned: {k} (5-fold, held-out molecules)"))

res = pd.DataFrame(results).set_index("ranking")
print((res * 100).round(1).assign(MRR=lambda d: (d.MRR / 100).round(3)).to_string())

# where is the current fusion losing? hit-rate by best-cosine bucket
best = df.groupby("q").spec_mod.max()
bucket = pd.cut(best, [-0.01, 0.3, 0.5, 0.7, 0.9, 0.95, 1.01])
cur = pd.Series(br < 1, index=qs_all).reindex(best.index)
d2 = df.assign(s=oof["hist-gbm"]); d2["r"] = d2.groupby("q")["s"].rank(ascending=False, method="first") - 1
gb_hit = (d2[d2.y == 1].set_index("q")["r"] < 1).reindex(best.index).fillna(False)
print("\nTop-1 hit rate by best-cosine bucket (n, current fusion, learned hist-gbm):")
print(pd.DataFrame({"n": best.groupby(bucket, observed=True).size(),
                    "current": (cur.groupby(bucket, observed=True).mean() * 100).round(1),
                    "learned": (gb_hit.groupby(bucket, observed=True).mean() * 100).round(1)}).to_string())

In [ ]:
import joblib
best_name = res.loc[[i for i in res.index if i.startswith("learned")], "MRR"].idxmax()
use_gbm = "hist-gbm" in best_name
print("Best learned model:", best_name)
if use_gbm:
    final = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.08, max_iter=250, random_state=SEED)
else:
    final = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))
final.fit(X, y)
out_path = os.path.join(WORKING_DIR, "reranker.joblib")
joblib.dump({"model": final, "features": FEATURE_NAMES}, out_path)
print("Saved:", out_path, "| features:", FEATURE_NAMES)
if not use_gbm:
    lr = final.named_steps["logisticregression"]
    print("Logistic coefficients (standardised):", dict(zip(FEATURE_NAMES, np.round(lr.coef_[0], 3))))
print("\nNext: upload reranker.joblib as a Kaggle dataset named 'casmi26-reranker' and attach it to the "
      "inference notebook (path is already in RERANKER_PATHS).")